In [2]:
companies = [
    "Nvidia",
    "Apple",
    "Alphabet",
    "Microsoft",
    "Amazon",
    "Broadcom",
    "Meta",
    "Tesla",
    "Oracle",
    "Advanced Micro Devices",
    "JPMorgan Chase",
    "Bank of America",
    "Citigroup",
    "Wells Fargo",
    "Goldman Sachs",
    "Morgan Stanley",
    "BlackRock",
    "American Express",
    "Charles Schwab",
    "Mastercard"
]

In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

values_block = "\n".join([f'"{c}"@en' for c in companies])

query = f"""
SELECT DISTINCT ?inputName ?company ?companyLabel ?ticker ?industryLabel ?countryLabel ?exchangeLabel
WHERE {{
  VALUES ?inputName {{
    {values_block}
  }}

  ?company rdfs:label ?inputName .
  ?company wdt:P31 ?instanceOf .

  FILTER(?instanceOf IN (wd:Q4830453, wd:Q6881511, wd:Q891723, wd:Q783794))

  OPTIONAL {{ ?company wdt:P249 ?ticker. }}
  OPTIONAL {{ ?company wdt:P452 ?industry. }}
  OPTIONAL {{ ?company wdt:P17 ?country. }}
  OPTIONAL {{ ?company wdt:P414 ?exchange. }}

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }}
}}
ORDER BY ?inputName
"""

sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

rows = []
for r in results["results"]["bindings"]:
    rows.append({
        "input_name": r["inputName"]["value"],
        "wikidata_id": r["company"]["value"].split("/")[-1],
        "wikidata_label": r.get("companyLabel", {}).get("value"),
        "ticker_wikidata": r.get("ticker", {}).get("value"),
        "industry": r.get("industryLabel", {}).get("value"),
        "country": r.get("countryLabel", {}).get("value"),
        "exchange": r.get("exchangeLabel", {}).get("value"),
    })

df_wikidata = pd.DataFrame(rows)
display(df_wikidata)

,input_name,wikidata_id,wikidata_label,ticker_wikidata,industry,country,exchange
0,Amazon,Q3884,Amazon,None,retail,United States,Nasdaq
1,Amazon,Q3884,Amazon,None,web service,United States,Nasdaq
2,Amazon,Q3884,Amazon,None,e-commerce,United States,Nasdaq
3,Amazon,Q3884,Amazon,None,web hosting service,United States,Nasdaq
4,Amazon,Q456085,Amazon,None,automotive industry,NaN,NaN
...,...,...,...,...,...,...,...
66,Tesla,Q1548225,Tesla,None,industrial manufacturing,Czech Republic,NaN
67,Tesla,Q78157177,Tesla,None,NaN,Czechoslovakia,NaN
68,Wells Fargo,Q744149,Wells Fargo,None,economics of banking,United States,New York Stock Exchange
69,Wells Fargo,Q744149,Wells Fargo,None,financial services,United States,New York Stock Exchange


In [4]:
import pandas as pd
from pathlib import Path

df_companies = pd.DataFrame({
    "canonical_name": [
        "Nvidia", "Apple", "Alphabet", "Microsoft", "Amazon", "Broadcom", "Meta", "Tesla", "Oracle", "AMD",
        "JPMorgan Chase", "Bank of America", "Citigroup", "Wells Fargo", "Goldman Sachs", "Morgan Stanley",
        "BlackRock", "American Express", "Charles Schwab", "Mastercard"
    ],
    "ticker": [
        "NVDA", "AAPL", "GOOGL", "MSFT", "AMZN", "AVGO", "META", "TSLA", "ORCL", "AMD",
        "JPM", "BAC", "C", "WFC", "GS", "MS", "BLK", "AXP", "SCHW", "MA"
    ],
    "sector": [
        "Tech", "Tech", "Tech", "Tech", "Tech", "Tech", "Tech", "Tech", "Tech", "Tech",
        "Financials", "Financials", "Financials", "Financials", "Financials", "Financials",
        "Financials", "Financials", "Financials", "Financials"
    ]
})

name_map = {
    "Advanced Micro Devices": "AMD"
}

df_wikidata["canonical_name"] = df_wikidata["input_name"].replace(name_map)

df_final = df_companies.merge(
    df_wikidata.drop(columns=["input_name"]).drop_duplicates(subset=["canonical_name"]),
    on="canonical_name",
    how="left"
)

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "companies.csv"
df_final.to_csv(out_path, index=False)

display(df_final)
print(f"Saved to {out_path}")

,canonical_name,ticker,sector,wikidata_id,wikidata_label,ticker_wikidata,industry,country,exchange
0,Nvidia,NVDA,Tech,Q182477,Nvidia,None,semiconductor industry,United States,Nasdaq
1,Apple,AAPL,Tech,NaN,NaN,NaN,NaN,NaN,NaN
2,Alphabet,GOOGL,Tech,NaN,NaN,NaN,NaN,NaN,NaN
3,Microsoft,MSFT,Tech,Q2283,Microsoft,None,software development,United States,Nasdaq
4,Amazon,AMZN,Tech,Q3884,Amazon,None,retail,United States,Nasdaq
5,Broadcom,AVGO,Tech,Q555925,Broadcom,None,semiconductor industry,United States,Nasdaq
6,Meta,META,Tech,Q380,Meta Platforms,None,information technology,United States,Nasdaq
7,Tesla,TSLA,Tech,Q478214,Tesla,None,automotive industry,United States,Nasdaq
8,Oracle,ORCL,Tech,Q30292006,Oracle,None,NaN,Spain,NaN
9,AMD,AMD,Tech,NaN,NaN,NaN,NaN,NaN,NaN


Saved to data\processed\companies.csv


In [5]:
import json

manual_aliases = {
    "Nvidia": ["Nvidia", "NVIDIA", "NVIDIA Corporation", "NVDA"],
    "Apple": ["Apple", "Apple Inc.", "AAPL"],
    "Alphabet": ["Alphabet", "Alphabet Inc.", "Google", "GOOGL"],
    "Microsoft": ["Microsoft", "Microsoft Corp.", "MSFT"],
    "Amazon": ["Amazon", "Amazon.com", "Amazon.com, Inc.", "AMZN"],
    "Broadcom": ["Broadcom", "Broadcom Inc.", "AVGO"],
    "Meta": ["Meta", "Meta Platforms", "Facebook", "META"],
    "Tesla": ["Tesla", "Tesla Inc.", "TSLA"],
    "Oracle": ["Oracle", "Oracle Corporation", "ORCL"],
    "AMD": ["AMD", "Advanced Micro Devices", "AMD Inc."],
    "JPMorgan Chase": ["JPMorgan Chase", "JPMorgan", "JPM"],
    "Bank of America": ["Bank of America", "BofA", "BAC"],
    "Citigroup": ["Citigroup", "Citi", "C"],
    "Wells Fargo": ["Wells Fargo", "WFC"],
    "Goldman Sachs": ["Goldman Sachs", "Goldman", "GS"],
    "Morgan Stanley": ["Morgan Stanley", "MS"],
    "BlackRock": ["BlackRock", "BLK"],
    "American Express": ["American Express", "AmEx", "AXP"],
    "Charles Schwab": ["Charles Schwab", "SCHW"],
    "Mastercard": ["Mastercard", "Mastercard Incorporated", "MA"],
}

df_final["aliases"] = df_final["canonical_name"].map(lambda x: json.dumps(manual_aliases.get(x, []), ensure_ascii=False))
df_final.to_csv(out_path, index=False)

display(df_final[["canonical_name", "ticker", "wikidata_id", "aliases"]])

,canonical_name,ticker,wikidata_id,aliases
0,Nvidia,NVDA,Q182477,"[""Nvidia"", ""NVIDIA"", ""NVIDIA Corporation"", ""NV..."
1,Apple,AAPL,NaN,"[""Apple"", ""Apple Inc."", ""AAPL""]"
2,Alphabet,GOOGL,NaN,"[""Alphabet"", ""Alphabet Inc."", ""Google"", ""GOOGL""]"
3,Microsoft,MSFT,Q2283,"[""Microsoft"", ""Microsoft Corp."", ""MSFT""]"
4,Amazon,AMZN,Q3884,"[""Amazon"", ""Amazon.com"", ""Amazon.com, Inc."", ""..."
5,Broadcom,AVGO,Q555925,"[""Broadcom"", ""Broadcom Inc."", ""AVGO""]"
6,Meta,META,Q380,"[""Meta"", ""Meta Platforms"", ""Facebook"", ""META""]"
7,Tesla,TSLA,Q478214,"[""Tesla"", ""Tesla Inc."", ""TSLA""]"
8,Oracle,ORCL,Q30292006,"[""Oracle"", ""Oracle Corporation"", ""ORCL""]"
9,AMD,AMD,NaN,"[""AMD"", ""Advanced Micro Devices"", ""AMD Inc.""]"


In [11]:
import pandas as pd
import json
from pathlib import Path

companies_path = Path("data/processed/companies.csv")
df = pd.read_csv(companies_path)

EVENT_TERMS = [
    "earnings",
    "revenue",
    "guidance",
    "lawsuit",
    "acquisition",
    "merger",
    "regulation",
    "downgrade",
    "layoffs",
    "product launch",
]

def parse_aliases(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    if isinstance(x, list):
        return x
    try:
        return json.loads(x)
    except Exception:
        return []

def clean_aliases_for_gdelt(aliases, canonical_name=None, ticker=None):
    cleaned = []
    seen = set()

    for a in aliases:
        if not a:
            continue
        a = str(a).strip()

        # Drop very short aliases
        if len(a) < 4:
            continue

        # Drop risky generic names unless they are the canonical name
        if a in {"Meta", "Apple", "Amazon", "Oracle"} and canonical_name != a:
            continue

        if a not in seen:
            cleaned.append(a)
            seen.add(a)

    # Always keep canonical name
    if canonical_name and canonical_name not in seen:
        cleaned.insert(0, canonical_name)
        seen.add(canonical_name)

    # Keep ticker only if long enough
    if ticker and len(str(ticker)) >= 4 and ticker not in seen:
        cleaned.append(str(ticker))

    return cleaned

def quote_term(term):
    term = str(term).strip().replace('"', '\\"')
    return f'"{term}"'

def build_company_query(aliases, event_terms):
    """
    Build a GDELT-style boolean query:
    ("Alias1" OR "Alias2") AND ("event1" OR "event2")
    """
    aliases = [a for a in aliases if str(a).strip()]
    event_terms = [e for e in event_terms if str(e).strip()]

    if not aliases:
        return None
    if not event_terms:
        return " OR ".join(quote_term(a) for a in aliases)

    alias_block = " OR ".join(quote_term(a) for a in aliases)
    event_block = " OR ".join(quote_term(e) for e in event_terms)

    return f"({alias_block}) AND ({event_block})"

# Build cleaned alias lists and query strings
df["aliases_list"] = df["aliases"].apply(parse_aliases)

df["gdelt_aliases"] = df.apply(
    lambda row: clean_aliases_for_gdelt(
        row["aliases_list"],
        canonical_name=row["canonical_name"],
        ticker=row["ticker"]
    ),
    axis=1
)

df["gdelt_query"] = df["gdelt_aliases"].apply(
    lambda xs: build_company_query(xs, EVENT_TERMS)
)

out_path = Path("data/processed/companies_with_gdelt_queries.csv")
df.to_csv(out_path, index=False)

print(f"Saved to {out_path}")
print(df[["canonical_name", "ticker", "gdelt_aliases", "gdelt_query"]].head(10).to_string(index=False))

Saved to data\processed\companies_with_gdelt_queries.csv
canonical_name ticker                                gdelt_aliases                                                                                                                                                                                                      gdelt_query
        Nvidia   NVDA   [Nvidia, NVIDIA, NVIDIA Corporation, NVDA]   ("Nvidia" OR "NVIDIA" OR "NVIDIA Corporation" OR "NVDA") AND ("earnings" OR "revenue" OR "guidance" OR "lawsuit" OR "acquisition" OR "merger" OR "regulation" OR "downgrade" OR "layoffs" OR "product launch")
         Apple   AAPL                    [Apple, Apple Inc., AAPL]                        ("Apple" OR "Apple Inc." OR "AAPL") AND ("earnings" OR "revenue" OR "guidance" OR "lawsuit" OR "acquisition" OR "merger" OR "regulation" OR "downgrade" OR "layoffs" OR "product launch")
      Alphabet  GOOGL     [Alphabet, Alphabet Inc., Google, GOOGL]     ("Alphabet" OR "Alphabet Inc." OR "Google" O

In [13]:
import time
import requests

GDELT_BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

HEADERS = {
    "User-Agent": "EP-GraphProject/1.0 (student project)"
}

def gdelt_request(params, max_retries=5, base_sleep=6):
    for attempt in range(max_retries):
        r = requests.get(GDELT_BASE_URL, params=params, headers=HEADERS, timeout=60)

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429:
            wait_time = base_sleep * (attempt + 1)
            print(f"429 rate limit hit. Sleeping {wait_time} seconds...")
            time.sleep(wait_time)
            continue

        print("Status:", r.status_code)
        print(r.text[:500])
        r.raise_for_status()

    raise RuntimeError("Too many retries on GDELT request")

In [15]:
import time, random

LAST_CALL_TS = 0.0

def wait_for_gdelt(min_interval=5.5):
    global LAST_CALL_TS
    now = time.time()
    elapsed = now - LAST_CALL_TS
    if elapsed < min_interval:
        time.sleep(min_interval - elapsed + random.uniform(0.1, 0.6))
    LAST_CALL_TS = time.time()

In [16]:
import requests

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": '"Apple Inc." AND ("earnings" OR "revenue" OR "guidance") AND sourcelang:english',
    "mode": "ArtList",
    "format": "json",
    "sort": "DateDesc",
    "maxrecords": 10,
    "timespan": "7d",
}

headers = {
    "User-Agent": "EP-GraphProject/1.0 (student project)"
}
wait_for_gdelt()
r = requests.get(url, params=params, headers=headers, timeout=60)

print("Status:", r.status_code)
print(r.text[:500])

if r.status_code == 200 and r.text.strip().startswith("{"):
    data = r.json()
    articles = data.get("articles", [])
    print("\nArticles returned:", len(articles))
    for a in articles[:5]:
        print("\nTITLE:", a.get("title"))
        print("LANG:", a.get("language"))
        print("DOMAIN:", a.get("domain"))
        print("DATE:", a.get("seendate"))
        print("URL:", a.get("url"))

Status: 200
{"articles": [ { "url": "https://www.thestar.com.my/tech/tech-news/2026/04/16/apple-google-offer-nudify-apps-despite-policies-against-them", "url_mobile": "", "title": "Apple , Google offer  nudify  apps despite policies against them", "seendate": "20260416T050000Z", "socialimage": "https://apicms.thestar.com.my/uploads/images/2026/04/16/3862005.jpg", "domain": "thestar.com.my", "language": "English", "sourcecountry": "Malaysia" },{ "url": "https://www.moneycontrol.com/technology/apple-google-of

Articles returned: 10

TITLE: Apple , Google offer  nudify  apps despite policies against them
LANG: English
DOMAIN: thestar.com.my
DATE: 20260416T050000Z
URL: https://www.thestar.com.my/tech/tech-news/2026/04/16/apple-google-offer-nudify-apps-despite-policies-against-them

TITLE: Apple , Google offer  nudify  apps despite policies against them
LANG: English
DOMAIN: moneycontrol.com
DATE: 20260416T034500Z
URL: https://www.moneycontrol.com/technology/apple-google-offer-nudify-apps-d